# Smoke test — Qwen2-Audio-7B-Instruct on the pilot stimuli (Kaggle)

Stage 0 of `report/Task.md`: before running the full pilot, confirm on a few samples that

1. the model loads and runs on audio + text inputs,
2. the **logit readout works**: almost all next-token probability lands on `A` or `B` (`mass_ab` ≈ 1), and
3. it **agrees with what the model actually says** in free-form generation (`greedy_agrees` ≈ 1).

If (2) or (3) fails, switch the pilot to `uq: self_consistency`.

**Notebook settings (right sidebar) before running:**
- **Accelerator → GPU T4 x2** (recommended: fp16 split across both GPUs, no quantization). A single P100 also works but falls back to 4-bit.
- **Internet → On** (needs a phone-verified Kaggle account) — required for `git clone`, `pip` and the model download.
- Private GitHub repo? **Add-ons → Secrets** → add `GITHUB_TOKEN` (a fine-grained token with read access to the repo) and tick it for this notebook.

The run covers 2 fictitious + 2 control facts × 8 conditions (conflict items in both A/B orders) = 36 stimuli; first-time model download is ~15 GB (~5 min).

## 1. GPU check

In [ ]:
import os, subprocess, torch
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2"
gpus = [torch.cuda.get_device_properties(i) for i in range(torch.cuda.device_count())]
VRAM_GB = sum(g.total_memory for g in gpus) / 1e9
for g in gpus:
    print(f"{g.name}: {g.total_memory / 1e9:.1f} GB")
# 7B in fp16 needs ~16 GB of weights plus activations; device_map="auto" splits it across GPUs
QUANTIZATION = "none" if VRAM_GB >= 20 else "4bit"
# native bf16 needs compute capability >= 8.0 (A100/L4); T4 / P100 only emulate it -> float16
DTYPE = "bfloat16" if min(torch.cuda.get_device_capability(i)[0] for i in range(len(gpus))) >= 8 else "float16"
print(f"total {VRAM_GB:.1f} GB -> quantization: {QUANTIZATION} | dtype: {DTYPE}")

## 2. Get the project code

Clones into `/kaggle/working/Thesis`, so `results/` ends up in the notebook's **Output** tab.
Set `REPO_URL` to your GitHub repo. For a private repo the token is read from Kaggle Secrets (never paste it into the notebook).

In [ ]:
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"   # <- edit
BRANCH = "main"
REPO = "/kaggle/working/Thesis"

url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
    print("using GITHUB_TOKEN from Kaggle Secrets")
except Exception:
    print("no GITHUB_TOKEN secret -> cloning as a public repo")

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)   # pick up new pushes
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO], check=True)
os.chdir(REPO)
assert os.path.exists("src/eval/run_model.py"), f"src/ not found under {REPO}"
print("working in", os.getcwd())
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 3. Install dependencies
Kaggle already has `torch` and `transformers`; this adds `bitsandbytes`, `edge-tts`, etc.

In [ ]:
!pip install -q -r requirements.txt
import transformers; print("transformers", transformers.__version__)

## 4. Kaggle config
Copies `configs/pilot.yaml` with the quantization / dtype chosen for this GPU.

In [ ]:
import yaml
cfg = yaml.safe_load(open("configs/pilot.yaml"))
cfg["model"]["quantization"] = QUANTIZATION
cfg["model"]["dtype"] = DTYPE
yaml.safe_dump(cfg, open("configs/kaggle.yaml", "w"), sort_keys=False)
print(yaml.safe_dump(cfg["model"]), "eval:", cfg["eval"]["uq"], "| greedy_check:", cfg["eval"]["greedy_check"])

## 5. Check the data
The stimuli and audio come from the repo; they are regenerated only if missing.

In [ ]:
import glob, json
if not os.path.exists("data/generated/stimuli.jsonl"):
    !python -m src.data_gen.text_variants
n_audio = len(glob.glob("data/audio/*.mp3"))
print("audio clips:", n_audio)
if n_audio == 0:
    !python -m src.data_gen.tts
    print("audio clips:", len(glob.glob("data/audio/*.mp3")))

In [ ]:
# listen to one clip to make sure the TTS is intelligible
from IPython.display import Audio, display
stim = json.loads(open("data/generated/stimuli.jsonl").readline())
print(stim["question"], "| audio says:", stim["audio_value"])
display(Audio(stim["audio"]))

## 6. Run the smoke test

`--limit 2` = the first 2 fictitious and 2 control facts, all 8 conditions.
Re-running the cell resumes where it stopped; delete `results/smoke` to start fresh.

In [ ]:
!python -m src.eval.run_model --config configs/kaggle.yaml --out results/smoke --limit 2

## 7. Read the result

In [ ]:
from IPython.display import Markdown
display(Markdown(open("results/smoke/summary.md").read()))

In [ ]:
import pandas as pd
rows = [json.loads(l) for l in open("results/smoke/predictions.jsonl")]
df = pd.DataFrame(rows)
cols = ["stim_id", "expected", "chosen_value", "confidence", "mass_ab", "top_token", "greedy_letter", "chosen_letter", "greedy_raw"]
pd.set_option("display.max_colwidth", 80)
df[[c for c in cols if c in df]]

### Verdict

- `mass_ab` ≥ 0.9 → the pre-filled prompt really makes the next token a letter.
- `greedy_agrees` ≥ 0.9 → the logit readout matches the model's own free-form answer.
- `greedy_format_ok` → how often the free-form answer is parseable JSON with a letter.

In [ ]:
mass = df["mass_ab"].mean() if "mass_ab" in df else float("nan")
has_greedy = "greedy_letter" in df
agree = (df["greedy_letter"] == df["chosen_letter"]).mean() if has_greedy else float("nan")
fmt_ok = df["greedy_letter"].notna().mean() if has_greedy else float("nan")
print(f"mass_ab          {mass:.3f}")
print(f"greedy_agrees    {agree:.3f}")
print(f"greedy_format_ok {fmt_ok:.3f}")
if mass >= 0.9 and agree >= 0.9:
    print("\nPASS: logit readout is reliable -> keep `uq: logit` for the full pilot.")
else:
    print("\nCHECK: logit readout looks unreliable -> inspect top_token / greedy_raw above;"
          " if confirmed, run the pilot with --uq self_consistency (section 8).")

## 8. (Only if the verdict failed) self-consistency fallback

Samples 10 free-form answers per stimulus instead of reading logits. Slower (~10× generation).

In [ ]:
RUN_FALLBACK = False
if RUN_FALLBACK:
    !python -m src.eval.run_model --config configs/kaggle.yaml --out results/smoke_sc --limit 2 --uq self_consistency
    display(Markdown(open("results/smoke_sc/summary.md").read()))

## 9. Keep the results

Everything under `/kaggle/working` shows up in the **Output** tab (download from there), and is kept
permanently if you **Save Version → Save & Run All**. This cell also writes one zip for a single download.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/smoke_results", "zip", "results/smoke")
print("download: Output tab -> smoke_results.zip")